# FEVER — LiteSemRAG 索引 + `chunk_cooccur_query` 检索评测（合并版）

本 notebook 把**索引构建**与**查询评测**合并到一个文件中，在 BEIR 风格的 **FEVER 子集**上验证 LiteSemRAG。
整体流程沿用 `scifact_litesemrag_index_eval.ipynb`，仅**数据加载**部分按 FEVER 的目录结构改写
（整理自 `/home/xiaoyue/ProtoGraphRAG/Fever_subset.ipynb`，把其中混乱的散落 cell 重新组织）。

- **数据加载**：读取 `corpus.jsonl` / `queries.jsonl`，gold 文档来自 BEIR 的 `qrels/test.tsv`
  （列为 `query-id`、`corpus-id`、`score`）。这与 SciFact 把 gold 放在 query `metadata` 不同。
- **索引/语义分配**：沿用 `Anchor+LLM` 语义分配（本地/API LLM 过滤 Wikidata 候选义项）。
- **查询/评测**：用 `chunk_cooccur_query` 做 chunk 级共现检索，指标为 **Recall@10**（per-query 与 micro 两种口径）
  与 **MRR@10**，并支持共现权重扫描。

> 说明：FEVER 子集 corpus 共 2000 篇、queries 共 500 条，且**每条 query 的 gold 文档都落在这 2000 篇内**，
> 因此默认 `NUM_DOCS=None`（全量 2000 篇）时 500 题全部可评测。FEVER 文档多为单句，普遍较短。

## 一、环境、路径与全部参数

In [1]:
from pathlib import Path
import os
import sys
import time

# 定位项目根目录，并切换工作目录、加入 Python 搜索路径
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch

# =============================================================================
# 一、运行环境
# =============================================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # 优先使用 CUDA

# =============================================================================
# 二、数据集（FEVER 子集 / BEIR 格式）
# =============================================================================
# 已下载的 FEVER 子集目录（含 corpus.jsonl / queries.jsonl / qrels/test.tsv）
FEVER_DIR = Path("/home/xiaoyue/ProtoGraphRAG/fever_subset")
QRELS_SPLIT = "test.tsv"   # gold 所在的 qrels 文件名

NUM_DOCS = None          # 索引的 corpus 子集大小（按 corpus.jsonl 顺序取前 NUM_DOCS 篇）；None 表示全量 2000 篇
INDEX_INCLUDE_TITLE = True  # 是否把文档标题拼进被索引正文（FEVER title 即实体名，信息量大，True 更利于检索）
PREVIEW_QID = None       # 预览样本时使用的 query id；None 表示自动取第一条可评测 query

# =============================================================================
# 三、索引构建与缓存
# =============================================================================
INDEX_OUTPUT_DIR = REPO_ROOT / "cache" / "fever_litesemrag_index"  # 索引与缓存文件输出目录
INDEX_RUN_TAG = "api"    # 缓存版本标签（区分 LLM 后端/实验配置）

SAVE_SPLIT_INDEX = False             # 是否分文件保存索引与 tensor
LOAD_SAVED_INDEX_IF_EXISTS = True    # 若已有缓存索引则直接加载、跳过重建
FORCE_REBUILD_INDEX = False          # 是否强制重建并覆盖已有缓存
VERIFY_LOAD_AFTER_SAVE = False       # 重建后是否重新加载校验

GRAPH_BATCH_SIZE = 64    # 并行索引时每批处理的 chunk 数
GRAPH_QUEUE_SIZE = 16    # 索引任务队列长度
GRAPH_CHUNK_SIZE = 256   # 文档切 chunk 时的最大字符数
MIN_OCCURRENCES_FOR_DESCRIPTION = 50  # token/phrase 至少出现多少次才进入语义描述分配流程

# =============================================================================
# 四、语义分配（Anchor + LLM）
# =============================================================================
SEMANTIC_ASSIGNMENT_METHOD = "Anchor-E-mutual [ce_fallback]"  # anchor 标注 + mutual-kNN 传播；不确定样本走 ce 兜底

# --- Anchor 专用 ---
ANCHOR_FRACTION = 0.15   # anchor 占某 token 全部 occurrence 的比例
ANCHOR_FFT_RATIO = 0.70  # anchor 中由 FFT 选出的比例（其余随机）
ANCHOR_MIN_COUNT = 2     # 每个 token 至少选取的 anchor 数
ANCHOR_MAX_COUNT = 15    # 每个 token 最多选取的 anchor 数
PROP_KNN_K = 8           # kNN 传播构图近邻数 k

# --- FFT 专用（Anchor 不确定样本 ce_fallback 时也会用到部分阈值）---
FFT_MAX_SAMPLES = 10
FFT_CONSENSUS_RATIO_THRESHOLD = 0.9
FFT_D1_D2_RATIO_THRESHOLD = 0.8
FFT_KNN_CHECK_K = 5
FFT_DANGER_NEIGHBOR_M = 10

# --- Wikidata 候选 / LLM 过滤 / 描述生成（共用）---
WIKIDATA_CANDIDATE_LIMIT = 5
USE_LLM_WIKIDATA = True             # 用 LLM 过滤 Wikidata 候选义项
LLM_WIKIDATA_USE_API = True         # False=本地 LLM，True=API
SEPARATE_LLM_CACHE_BY_BACKEND = True
SEM_DESCRIPTION_BATCH_SIZE = 32
SEM_DESCRIPTION_PROMPT_CONTEXT_MODE = "sentence_neighbors"
TAU_CONC = 0.88                     # s_mean 单义闸门

# =============================================================================
# 五、评测参数
# =============================================================================
TOP_K = 10                 # Recall@K / MRR@K 的 K
EVAL_MAX_QUERIES = None     # 评测 query 上限；None 表示全部可评测 query
PRINT_IMPORTANT_TOKENS = False  # 查询时是否打印调试 token 信息

# =============================================================================
# 六、调试 / 浏览
# =============================================================================
ENABLE_PHRASE_AUDIT = True          # 索引时记录短语抽取审计日志
CLEAR_PHRASE_AUDIT_CACHE_ON_REBUILD = True
MULTI_SEM_MIN_SEM_COUNT = 2         # 多义项浏览：至少有这么多 sem node 的 token 才展示
MULTI_SEM_MAX_TOKEN_NODES = 50
MULTI_SEM_MAX_SEMS_PER_TOKEN = 10
MULTI_SEM_MAX_SENTENCES_PER_SEM = 3
MULTI_SEM_MAX_EXAMPLES_PER_TOKEN = 10
MULTI_SEM_TOKEN_CONTAINS = None

print(f"Working directory: {REPO_ROOT}")
print(f"Device: {DEVICE}")

Working directory: /home/xiaoyue/LiteSemRAG
Device: cuda


In [2]:
import RAG_graph

DATASET_TAG = "all" if NUM_DOCS is None else str(NUM_DOCS)
INDEX_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INDEX_BASENAME = f"litesemrag_fever_{DATASET_TAG}"
if INDEX_RUN_TAG:
    INDEX_BASENAME = f"{INDEX_BASENAME}_{INDEX_RUN_TAG}"
INDEX_PKL_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}.pkl"
INDEX_TENSOR_PATH = Path(str(INDEX_PKL_PATH).replace(".pkl", "_tensors.pt"))
INDEX_DOCS_JSON_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_documents.json"
INDEX_META_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_meta.json"
PHRASE_AUDIT_CACHE_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_phrase_audit.jsonl"

# --- LLM 应答缓存（sqlite），按后端(API/本地)分离 ---
LLM_CACHE_DIR = REPO_ROOT / "cache"
LLM_BACKEND_TAG = "api" if LLM_WIKIDATA_USE_API else "local"

def _backend_cache_path(filename):
    path = LLM_CACHE_DIR / filename
    if SEPARATE_LLM_CACHE_BY_BACKEND:
        path = path.with_name(f"{path.stem}_{LLM_BACKEND_TAG}{path.suffix}")
    return path

LLM_CANDIDATE_FILTER_CACHE_PATH = _backend_cache_path("wikidata_definition_filter_cache.sqlite3")
LLM_SEMANTIC_LABELER_CACHE_PATH = _backend_cache_path("llm_semantic_label_cache.sqlite3")

ANCHOR_METHOD_SPEC = RAG_graph._parse_anchor_method(SEMANTIC_ASSIGNMENT_METHOD)
IS_ANCHOR_METHOD = ANCHOR_METHOD_SPEC is not None
USE_LLM_SEMANTIC_LABELER = SEMANTIC_ASSIGNMENT_METHOD == "FFT-LLM"

GRAPH_CONFIG = dict(
    min_occurrences_for_description=MIN_OCCURRENCES_FOR_DESCRIPTION,
    chunk_size=GRAPH_CHUNK_SIZE,
    device=DEVICE,
    sem_assignment_method=SEMANTIC_ASSIGNMENT_METHOD,
    consensus_ratio_threshold=FFT_CONSENSUS_RATIO_THRESHOLD,
    fft_max_samples=FFT_MAX_SAMPLES,
    fft_d1_d2_ratio_threshold=FFT_D1_D2_RATIO_THRESHOLD,
    fft_knn_check_k=FFT_KNN_CHECK_K,
    fft_danger_neighbor_m=FFT_DANGER_NEIGHBOR_M,
    anchor_fraction=ANCHOR_FRACTION,
    anchor_fft_ratio=ANCHOR_FFT_RATIO,
    anchor_min_count=ANCHOR_MIN_COUNT,
    anchor_max_count=ANCHOR_MAX_COUNT,
    prop_knn_k=PROP_KNN_K,
    tau_conc=TAU_CONC,
    use_llm_candidate_filter=USE_LLM_WIKIDATA,
    use_llm_semantic_labeler=USE_LLM_SEMANTIC_LABELER,
    llm_candidate_filter_use_api=LLM_WIKIDATA_USE_API,
    llm_candidate_filter_cache_path=str(LLM_CANDIDATE_FILTER_CACHE_PATH),
    llm_semantic_labeler_cache_path=str(LLM_SEMANTIC_LABELER_CACHE_PATH),
    sem_description_prompt_context_mode=SEM_DESCRIPTION_PROMPT_CONTEXT_MODE,
    phrase_audit_enabled=ENABLE_PHRASE_AUDIT,
    phrase_audit_cache_path=str(PHRASE_AUDIT_CACHE_PATH),
)


def apply_semantic_assignment_runtime_settings(graph):
    graph.sem_description_candidate_limit = WIKIDATA_CANDIDATE_LIMIT
    graph.llm_candidate_filter_candidate_limit = WIKIDATA_CANDIDATE_LIMIT
    graph.use_llm_semantic_labeler = USE_LLM_SEMANTIC_LABELER
    graph.sem_description_batch_size = SEM_DESCRIPTION_BATCH_SIZE
    graph.sem_description_prompt_context_mode = SEM_DESCRIPTION_PROMPT_CONTEXT_MODE
    return graph


def write_index_metadata(graph=None, path=INDEX_META_PATH):
    import json as _json
    from datetime import datetime
    meta = {
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "dataset": "fever",
        "num_docs": NUM_DOCS,
        "index_include_title": INDEX_INCLUDE_TITLE,
        "semantic_assignment_method": SEMANTIC_ASSIGNMENT_METHOD,
        "graph_config": {k: (str(v) if isinstance(v, Path) else v) for k, v in GRAPH_CONFIG.items()},
        "index_pkl": str(INDEX_PKL_PATH),
    }
    if graph is not None:
        meta["graph_stats"] = {
            "docs": len(graph.doc_nodes),
            "chunks": len(graph.chunk_nodes),
            "tokens": len(graph.token_nodes),
            "phrase_tokens": len(graph.phrase_token_nodes),
            "sem_nodes": len(graph.sem_nodes),
        }
    Path(path).write_text(_json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved index metadata to: {path}")
    return meta


print(f"Index pickle path: {INDEX_PKL_PATH}")
print(f"Semantic assignment method: {SEMANTIC_ASSIGNMENT_METHOD} (anchor={IS_ANCHOR_METHOD})")
print(f"LLM backend tag: {LLM_BACKEND_TAG}")
print(f"Load saved index if available: {LOAD_SAVED_INDEX_IF_EXISTS}; Force rebuild: {FORCE_REBUILD_INDEX}")

Index pickle path: /home/xiaoyue/LiteSemRAG/cache/fever_litesemrag_index/litesemrag_fever_all_api.pkl
Semantic assignment method: Anchor-E-mutual [ce_fallback] (anchor=True)
LLM backend tag: api
Load saved index if available: True; Force rebuild: False


## 二、加载 FEVER 子集数据集

整理自 `Fever_subset.ipynb`：从 `corpus.jsonl` 读语料、从 `queries.jsonl` 读 claim、
从 `qrels/test.tsv` 读 gold 文档（BEIR qrels 三列：`query-id`、`corpus-id`、`score`）。
`corpus._id` 即文档名（与 qrels 的 `corpus-id` 对齐），FEVER 的 `title` 是实体名、`text` 为单句证据。

In [3]:
import json
import csv

def ensure_fever_dir():
    """返回可用的 FEVER 子集目录；缺文件则报错（该子集为本地自建，非 BEIR 自动下载）。"""
    if (FEVER_DIR / "corpus.jsonl").exists() and (FEVER_DIR / "queries.jsonl").exists() \
            and (FEVER_DIR / "qrels" / QRELS_SPLIT).exists():
        return FEVER_DIR
    raise FileNotFoundError(
        f"FEVER 子集不完整，缺少 corpus.jsonl / queries.jsonl / qrels/{QRELS_SPLIT}: {FEVER_DIR}"
    )


def load_corpus(path):
    """docid -> {title, text}（docid 统一转字符串）。"""
    corpus = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            corpus[str(obj["_id"])] = {"title": obj.get("title", ""), "text": obj.get("text", "")}
    return corpus


def load_queries(path):
    """qid -> claim 文本（qid 统一转字符串）。"""
    queries = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            queries[str(obj["_id"])] = obj["text"]
    return queries


def load_qrels(path):
    """qid -> gold 文档 id 集合（BEIR qrels TSV，列: query-id, corpus-id, score；只保留 score>0）。"""
    gold_docs = {}
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader, None)  # 跳过表头
        for row in reader:
            if len(row) < 3:
                continue
            qid, docid, score = row[0], row[1], row[2]
            try:
                if int(score) <= 0:
                    continue
            except ValueError:
                pass
            gold_docs.setdefault(str(qid), set()).add(str(docid))
    return gold_docs


data_dir = ensure_fever_dir()
print(f"FEVER dir: {data_dir}")

corpus = load_corpus(data_dir / "corpus.jsonl")
queries = load_queries(data_dir / "queries.jsonl")
gold_docs = load_qrels(data_dir / "qrels" / QRELS_SPLIT)
print(f"Corpus docs: {len(corpus)}  |  Queries: {len(queries)}  |  Qrels qids: {len(gold_docs)}")

# --- 构建索引子集：按 corpus 顺序取前 NUM_DOCS 篇，title 用 docid，doc_name 即 docid ---
corpus_items = list(corpus.items())  # [(docid, {title, text}), ...]
if NUM_DOCS is not None:
    corpus_items = corpus_items[:NUM_DOCS]

corpus_input = []
for docid, value in corpus_items:
    text = value["text"]
    if INDEX_INCLUDE_TITLE and value.get("title"):
        text = f"{value['title']}. {text}"  # 把标题(实体名)拼进正文，提升检索可召回性
    corpus_input.append({"title": docid, "text": text})  # title=docid -> 索引后 doc_name=docid

indexed_doc_ids = {item["title"] for item in corpus_input}
print(f"Indexed subset size: {len(corpus_input)} docs")

# --- 评测集：只保留 gold 非空、且 gold 全部落在索引子集内的 query ---
eval_items = []  # [(qid, query_text, gold_set), ...]
for qid, text in queries.items():
    g = gold_docs.get(qid, set())
    if not g:
        continue
    if not g.issubset(indexed_doc_ids):
        continue
    eval_items.append((qid, text, g))

if EVAL_MAX_QUERIES is not None:
    eval_items = eval_items[:EVAL_MAX_QUERIES]
print(f"Evaluable queries (gold 全在索引子集内): {len(eval_items)}")

FEVER dir: /home/xiaoyue/ProtoGraphRAG/fever_subset
Corpus docs: 2000  |  Queries: 500  |  Qrels qids: 500
Indexed subset size: 2000 docs
Evaluable queries (gold 全在索引子集内): 500


In [4]:
# 预览一条可评测样本
if eval_items:
    if PREVIEW_QID is not None and any(qid == str(PREVIEW_QID) for qid, _, _ in eval_items):
        qid, qtext, gold = next(item for item in eval_items if item[0] == str(PREVIEW_QID))
    else:
        qid, qtext, gold = eval_items[0]
    print(f"QID: {qid}")
    print(f"Claim: {qtext}")
    print(f"Gold doc ids: {gold}")
    for docid in gold:
        print(f"\n[{docid}] title: {corpus[docid]['title']}")
        print(corpus[docid]['text'][:400])
else:
    print("没有可评测 query，请调大 NUM_DOCS。")

QID: 174601
Claim: Artpop was reviewed by dog critics.
Gold doc ids: {'Artpop'}

[Artpop] title: Artpop
Artpop ( stylized as ARTPOP ) is the third studio album by American singer Lady Gaga , released on November 6 , 2013 , by Streamline and Interscope Records . Gaga began planning the project in 2011 , shortly after the launch of her second studio album , Born This Way . Work continued until 2013 while Gaga was traveling for her Born This Way Ball concert tour and recovering from surgery for an inju


## 三、构建 / 加载索引

In [5]:
def index_cache_exists():
    if SAVE_SPLIT_INDEX:
        return INDEX_PKL_PATH.exists() and INDEX_TENSOR_PATH.exists()
    return INDEX_PKL_PATH.exists()

INDEX_WAS_REBUILT = False
INDEX_CACHE_AVAILABLE = index_cache_exists()

if LOAD_SAVED_INDEX_IF_EXISTS and INDEX_CACHE_AVAILABLE and not FORCE_REBUILD_INDEX:
    print(f"Loading saved index from: {INDEX_PKL_PATH}")
    if SAVE_SPLIT_INDEX:
        graph_database = RAG_graph.LiteSemRAG.load_data_split(str(INDEX_PKL_PATH))
    else:
        graph_database = RAG_graph.LiteSemRAG.load_data(str(INDEX_PKL_PATH))
    graph_database.json_path = str(INDEX_DOCS_JSON_PATH)
    apply_semantic_assignment_runtime_settings(graph_database)
    if ENABLE_PHRASE_AUDIT:
        graph_database.enable_phrase_audit(str(PHRASE_AUDIT_CACHE_PATH))
else:
    if FORCE_REBUILD_INDEX and INDEX_CACHE_AVAILABLE:
        print("Force rebuild enabled; existing cache will be overwritten after indexing.")
    elif LOAD_SAVED_INDEX_IF_EXISTS:
        print("No complete saved index cache found; building a new index.")
    else:
        print("Saved-index loading disabled; building a new index.")

    if ENABLE_PHRASE_AUDIT and CLEAR_PHRASE_AUDIT_CACHE_ON_REBUILD and PHRASE_AUDIT_CACHE_PATH.exists():
        PHRASE_AUDIT_CACHE_PATH.unlink()
        print(f"Removed old phrase audit cache: {PHRASE_AUDIT_CACHE_PATH}")

    graph_database = RAG_graph.LiteSemRAG(**GRAPH_CONFIG)
    graph_database.json_path = str(INDEX_DOCS_JSON_PATH)
    apply_semantic_assignment_runtime_settings(graph_database)

    build_start = time.time()
    graph_database.index_json(
        corpus_input,
        batch_size=GRAPH_BATCH_SIZE,
        queue_size=GRAPH_QUEUE_SIZE,
        sample_count=len(corpus_input),
    )
    graph_database.finalize()
    print(f"索引+finalize 耗时: {time.time() - build_start:.1f}s")
    graph_database.print_memory_size()

    if SAVE_SPLIT_INDEX:
        graph_database.save_data_split(str(INDEX_PKL_PATH))
        print(f"Saved split index to: {INDEX_PKL_PATH}")
    else:
        graph_database.save_data(str(INDEX_PKL_PATH))
        print(f"Saved index to: {INDEX_PKL_PATH}")
    write_index_metadata(graph_database)
    INDEX_WAS_REBUILT = True

print("Graph stats:")
print(f"  loaded from cache: {not INDEX_WAS_REBUILT}")
print(f"  docs: {len(graph_database.doc_nodes)}")
print(f"  chunks: {len(graph_database.chunk_nodes)}")
print(f"  tokens: {len(graph_database.token_nodes)}")
print(f"  phrase tokens: {len(graph_database.phrase_token_nodes)}")
print(f"  sem nodes: {len(graph_database.sem_nodes)}")
qdb = graph_database.query_database
print(f"  query database shape: {None if qdb is None else tuple(qdb.shape)}")

No complete saved index cache found; building a new index.
Loading text encoder models in device: GPU


/home/xiaoyue/anaconda3/envs/llm_graph/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


CPU preprocessed: 2000/2000 | GPU encoded: 2000/2000 | CPU processed: 2000/2000
[592.0517s] Index 2000 documents. Index pipeline time: 592.0517s
[592.0577s] Finalize started.
[592.0599s] Computed average chunk length.
[592.1446s] Removed 2149 empty placeholder token nodes.
finalize_token_nodes: 163 potential multi-sense node(s) (occurrences >= min_occurrences_for_description=50), 29595 basic node(s).
  Building potential multi-sense nodes (most time-consuming stage)...
multi_sense_node_build: 163/163 | remaining:   0
basic_node_build: 29595/29595 | remaining:     0
[1365.7105s] Finished token node finalization.
sem-build funnel (min_occurrences_for_description=50):
  1. 通过出现次数闸门进入完整建节点路径: 163 (其中 entity/原子短语默认单义: 33)
  2. 参与 S_mean 判定(非 entity): 130 -> 因 S_mean 过高被筛为单义: 22
  3. 通过 S_mean 进入消歧路径: 108 -> 最终切出多个语义: 48
[1365.9884s] Finished building modifier postings.
[1366.0576s] Finished assigning token and sem IDF.
[1366.1904s] Finished computing sem BM25.
[1366.3490s] Finished building

### （可选）多义项浏览

查看被切分出多个语义节点的 token，人工核对 Anchor+LLM 的语义分配质量。

In [6]:
from IPython.display import display

multi_sem_token_nodes = [
    tn for tn in graph_database.token_nodes
    if len(tn.sem_node_list) >= MULTI_SEM_MIN_SEM_COUNT
]
print(
    f"Token nodes with >= {MULTI_SEM_MIN_SEM_COUNT} sem nodes: "
    f"{len(multi_sem_token_nodes)} / {len(graph_database.token_nodes)}"
)
display(
    graph_database.show_multi_sem_token_nodes(
        min_sem_count=MULTI_SEM_MIN_SEM_COUNT,
        max_sentences_per_sem=MULTI_SEM_MAX_SENTENCES_PER_SEM,
        as_html=True,
        token_contains=MULTI_SEM_TOKEN_CONTAINS,
        sort_by="sem_count",
        max_token_nodes=MULTI_SEM_MAX_TOKEN_NODES,
        max_sems_per_token=MULTI_SEM_MAX_SEMS_PER_TOKEN,
        max_examples_per_token=MULTI_SEM_MAX_EXAMPLES_PER_TOKEN,
        open_details=False,
    )
)

Token nodes with >= 2 sem nodes: 48 / 29758


## 四、评测：`chunk_cooccur_query` 的 Recall@10 / MRR@10

对每条可评测 query 调用 `chunk_cooccur_query`，把检索回的 chunk 映射成所属文档 id（去重保序），与该题 gold 对比：

- **Recall@K (per-query)**：每题 `命中 gold 数 / gold 总数`，再对所有题取平均。
- **Recall@K (micro)**：`所有题命中 gold 总数 / 所有题 gold 总数`。
- **MRR@K**：第一个命中 gold 的检索名次倒数，再取平均。

In [7]:
from utils import mrr_for_one_query_titles


def retrieved_docids_for_query(question, lambda_boost, top_k_pair_boosts):
    """返回检索到的文档 id 列表（按检索名次去重保序）。"""
    _, retrieved_chunk_ids, _ = graph_database.chunk_cooccur_query(
        question,
        top_k_chunk=TOP_K,
        top_k_pair_boosts=top_k_pair_boosts,
        lambda_boost=lambda_boost,
        print_important_tokens=PRINT_IMPORTANT_TOKENS,
    )
    docids = []
    seen = set()
    for cid in retrieved_chunk_ids:
        docid = graph_database.chunk_nodes[cid].doc_node.doc_name
        if docid not in seen:
            seen.add(docid)
            docids.append(docid)
    return docids


def evaluate_config(lambda_boost, top_k_pair_boosts):
    """在全部 eval_items 上跑一组权重，返回指标 dict。"""
    per_query_recall_sum = 0.0
    total_gold_hits = 0
    total_gold = 0
    mrr_sum = 0.0
    counted = 0

    start = time.time()
    for qid, qtext, gold_set in eval_items:
        counted += 1
        ranked = retrieved_docids_for_query(qtext, lambda_boost, top_k_pair_boosts)[:TOP_K]
        retrieved_set = set(ranked)
        num_hit = len(gold_set & retrieved_set)
        per_query_recall_sum += num_hit / len(gold_set)
        total_gold_hits += num_hit
        total_gold += len(gold_set)
        mrr_sum += mrr_for_one_query_titles(ranked, list(gold_set), k=TOP_K)

    elapsed = time.time() - start
    return {
        "lambda_boost": lambda_boost,
        "top_k_pair_boosts": top_k_pair_boosts,
        f"recall@{TOP_K}_per_query": per_query_recall_sum / counted if counted else 0.0,
        f"recall@{TOP_K}_micro": total_gold_hits / total_gold if total_gold else 0.0,
        f"mrr@{TOP_K}": mrr_sum / counted if counted else 0.0,
        "n": counted,
        "sec": elapsed,
    }

In [8]:
from RAG_graph import COOCCUR_LAMBDA, COOCCUR_TOP_K_PAIR_BOOSTS

# 用 chunk_cooccur_query 的默认权重跑一遍全量评测
metrics = evaluate_config(COOCCUR_LAMBDA, COOCCUR_TOP_K_PAIR_BOOSTS)
print(f"默认权重 (λ={COOCCUR_LAMBDA}, top_k_pair={COOCCUR_TOP_K_PAIR_BOOSTS})，共 {metrics['n']} 题:")
print(f"  Recall@{TOP_K} (per-query): {metrics[f'recall@{TOP_K}_per_query']:.4f}")
print(f"  Recall@{TOP_K} (micro)    : {metrics[f'recall@{TOP_K}_micro']:.4f}")
print(f"  MRR@{TOP_K}               : {metrics[f'mrr@{TOP_K}']:.4f}")
print(f"  耗时: {metrics['sec']:.1f}s（{metrics['sec']/max(metrics['n'],1):.3f}s/题）")

默认权重 (λ=0.3, top_k_pair=3)，共 500 题:
  Recall@10 (per-query): 0.8838
  Recall@10 (micro)    : 0.8485
  MRR@10               : 0.8149
  耗时: 34.0s（0.068s/题）


## 五、（可选）共现权重扫描

扫描 `chunk_cooccur_query` 的 `lambda_boost`（共现加成强度）与 `top_k_pair_boosts`（每 chunk 取多少最强 pair 求平均）。

`final_score(chunk) = BaseEvidence × (1 + λ × PairBoost)`；`λ=0` 退化为纯 BaseEvidence 基线。

In [9]:
import pandas as pd

# 每个元素 = (lambda_boost, top_k_pair_boosts)
WEIGHT_CONFIGS = [
    (0.0, 3),   # 基线：关闭共现加成
    (0.1, 3),
    (0.3, 3),   # 当前默认
    (0.5, 3),
    (1.0, 3),
    (0.3, 1),
    (0.3, 5),
]

results = []
sweep_start = time.time()
for idx, (lam, topk) in enumerate(WEIGHT_CONFIGS):
    m = evaluate_config(lam, topk)
    results.append(m)
    print(
        f"[{idx + 1}/{len(WEIGHT_CONFIGS)}] λ={lam}, top_k_pair={topk} | "
        f"R@{TOP_K}(pq)={m[f'recall@{TOP_K}_per_query']:.4f} "
        f"R@{TOP_K}(micro)={m[f'recall@{TOP_K}_micro']:.4f} "
        f"MRR@{TOP_K}={m[f'mrr@{TOP_K}']:.4f} ({m['sec']:.1f}s)"
    )
print(f"\n扫描总耗时: {time.time() - sweep_start:.1f}s")

results_df = pd.DataFrame(results).sort_values(
    f"recall@{TOP_K}_per_query", ascending=False
).reset_index(drop=True)
results_df

[1/7] λ=0.0, top_k_pair=3 | R@10(pq)=0.8825 R@10(micro)=0.8485 MRR@10=0.8070 (34.0s)
[2/7] λ=0.1, top_k_pair=3 | R@10(pq)=0.8818 R@10(micro)=0.8468 MRR@10=0.8101 (33.8s)
[3/7] λ=0.3, top_k_pair=3 | R@10(pq)=0.8838 R@10(micro)=0.8485 MRR@10=0.8149 (33.9s)
[4/7] λ=0.5, top_k_pair=3 | R@10(pq)=0.8838 R@10(micro)=0.8485 MRR@10=0.8141 (33.9s)
[5/7] λ=1.0, top_k_pair=3 | R@10(pq)=0.8848 R@10(micro)=0.8502 MRR@10=0.8210 (34.0s)
[6/7] λ=0.3, top_k_pair=1 | R@10(pq)=0.8838 R@10(micro)=0.8485 MRR@10=0.8149 (33.9s)
[7/7] λ=0.3, top_k_pair=5 | R@10(pq)=0.8838 R@10(micro)=0.8485 MRR@10=0.8149 (34.1s)

扫描总耗时: 237.6s


,lambda_boost,top_k_pair_boosts,recall@10_per_query,recall@10_micro,mrr@10,n,sec
0,1.0,3,0.884800,0.850168,0.820995,500,34.015541
1,0.5,3,0.883800,0.848485,0.814083,500,33.942268
2,0.3,3,0.883800,0.848485,0.814906,500,33.869432
3,0.3,1,0.883800,0.848485,0.814906,500,33.925339
4,0.3,5,0.883800,0.848485,0.814906,500,34.074914
5,0.0,3,0.882467,0.848485,0.807021,500,33.967847
6,0.1,3,0.881800,0.846801,0.810117,500,33.847175


## 六、（可选）单样本检查

In [10]:
inspect_index = 0
inspect_lambda = 0.3
inspect_top_k_pair = 3

qid, qtext, gold = eval_items[inspect_index]
print("QID:", qid)
print("Claim:", qtext)
print("Gold doc ids:", gold)
for docid in gold:
    print(f"  [{docid}] {corpus[docid]['title']}")

retrieved_chunk, retrieved_chunk_ids, _ = graph_database.chunk_cooccur_query(
    qtext,
    top_k_chunk=TOP_K,
    top_k_pair_boosts=inspect_top_k_pair,
    lambda_boost=inspect_lambda,
    print_important_tokens=True,
)

print("\n--- 检索到的 chunk（按名次）---")
for rank, (cid, text) in enumerate(zip(retrieved_chunk_ids, retrieved_chunk)):
    docid = graph_database.chunk_nodes[cid].doc_node.doc_name
    hit = "  <== GOLD" if docid in gold else ""
    print(f"[{rank}] chunk_id={cid} | doc={docid}{hit}")
    print(f"     {text[:200]}")

QID: 174601
Claim: Artpop was reviewed by dog critics.
Gold doc ids: {'Artpop'}
  [Artpop] Artpop
important ents: ['artpop']
important phrases: ['dog critics']
important tokens: ['dog', 'critic']
query tokens: ['artpop', 'dog critics', 'critic', 'dog']
low level tokens: ['artpop(exact matched)', 'remote electrodes(sim matched)', 'critic(exact matched)', 'dog(exact matched)']
high level tokens: [['N/A'], 'middle-aged critic(partial matched)', 'regular dog(partial matched)']
{1710: ['Token:artpop,Score:12.0961'], 258: ['Token:remote electrodes,Score:9.9686'], 31: ['Token:critic,Score:3.1941'], 53: ['Token:critic,Score:2.5509'], 87: ['Token:critic,Score:4.4956'], 106: ['Token:critic,Score:3.3181'], 151: ['Token:critic,Score:2.1736'], 327: ['Token:critic,Score:2.1152'], 328: ['Token:critic,Score:2.6426'], 361: ['Token:critic,Score:3.0729'], 397: ['Token:critic,Score:3.8262'], 455: ['Token:critic,Score:5.3592'], 470: ['Token:critic,Score:2.5450'], 525: ['Token:critic,Score:3.8049'], 542: ['